# 3 - Feederplots (template)

Plot feeder/exit average counts for one season using the `bb_metrics` package (`POLO` model, CLAHE). Swap the config import below for your season.

1. **Hourly average counts per hive** + weather panel, with treatment-day shading.
2. **Daily average counts** lineplot per hive.

In [ ]:
import glob, os
from datetime import timedelta

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import bb_metrics
cfg = bb_metrics.load_config('/path/to/your/season_config.py')  # replace with your season config
import bb_metrics
import bb_metrics.displayfunctions as bp
import bb_metrics.datafunctions as dfunc
from bb_metrics import feedercams as fc

bb_metrics.set_config(cfg)
bp.init(cfg)
dfunc.init(cfg)
bd = cfg

sns.set_style("whitegrid")

## Load CLAHE average counts (feeder + exit)

In [ ]:
dfcounts_feeder = fc.load_avgcounts("feedercam", clahe=True, cfg=bd)
dfcounts_exit   = fc.load_avgcounts("exitcam",   clahe=True, cfg=bd)

# Konstanz: feeder data is stored as "outdoorcam" and there is one hive, so the
# file token (glob) and the row mapping differ -- pass cam_to_hive explicitly:
#   dfcounts_feeder = fc.load_avgcounts("outdoorcam", cfg=bd,
#                                       cam_to_hive=fc.cam_hive_map("feedercam", bd))

for name, df in [("feeder", dfcounts_feeder), ("exit", dfcounts_exit)]:
    print(
        f"{name}: {df.shape[0]} rows, "
        f"{df['video_start_timestamp'].min()} -> {df['video_start_timestamp'].max()}, "
        f"hives={sorted(df['hive'].unique())}"
    )

## Treatment days (day-of-week)

Tuesday + Wednesday are treatment days; feeders **A** and **D** receive the pesticide.
Edit `treatment_weekdays` / `treated_feeders` to re-select. Builds a `treat_df`
(columns `[feeder, start, end]`) compatible with `bp.shade_treatments` — control
feeders (B, C) get no bands.

In [ ]:
# Treatment days are defined inline (experiment-specific; not part of the package).
# --- configurable day-of-week treatment scheme ---
treatment_weekdays = [1, 2]       # Mon=0 ... Tue=1, Wed=2
treated_feeders    = ["A", "D"]   # feeders receiving pesticide treatment


def build_treat_df_from_weekdays(start, end, feeders, weekdays, tz="Europe/Berlin"):
    """treat_df with columns [feeder, start, end]; one full-day band
    (00:00 -> next 00:00, Berlin local) per matching weekday, per treated feeder."""
    days = pd.date_range(pd.Timestamp(start).normalize(), pd.Timestamp(end).normalize(), freq="D")
    rows = []
    for f in feeders:
        for d in days:
            if d.weekday() in weekdays:
                s = d.tz_localize(tz) if d.tz is None else d.tz_convert(tz)
                rows.append({"feeder": f"Feeder {f}", "start": s, "end": s + pd.Timedelta(days=1)})
    return pd.DataFrame(rows)


treat_df = build_treat_df_from_weekdays(bd.startday, bd.endday, treated_feeders, treatment_weekdays)
print(treat_df.head(12))
print(f"{len(treat_df)} treatment-day bands across feeders {treated_feeders}")

## Hourly average counts per hive + weather

Counts are aggregated to hourly values with `get_feedercam_hourly_average`: each
video's per-frame average is weighted by the seconds it actually covers within the
hour (clipped at hour boundaries) and divided by 3600. This is the **time-weighted /
coverage-aware estimate** (the master hour grid is the weather index), unbiased by
uneven video cadence — preferred over a naive mean of per-video averages.

Everything is kept on one **Europe/Berlin** timeline so the Tue/Wed treatment bands
land on the bees' true local days. One figure per camera (feeder/exit) x metric
(total/tagged/untagged): four stacked per-hive panels (shared y) + a weather panel.
No-data days show as line breaks; red bands mark treatment days on the A and D panels.

In [ ]:
tz = "Europe/Berlin"
video_duration_seconds = 30   # feeder-cam clip length used for the time-weighting

start = pd.Timestamp(bd.startday).tz_localize(tz)
end = pd.Timestamp(bd.endday).tz_localize(tz)

# Weather on the same Berlin timeline as the counts (also the master hour grid).
dfweather_hourly = dfunc.get_weather_data(
    bd.weather_station_id, start, end + timedelta(days=1), data_type="hourly", tz=tz
)

metrics = [("totalcounts", "Total"), ("taggedcounts", "Tagged"), ("untaggedcounts", "Untagged")]
cameras = [("feeder", dfcounts_feeder), ("exit", dfcounts_exit)]

for which, dfc in cameras:
    dfc = dfc.copy()
    # day-level coverage: a hive/day with any video counts as "has data"
    dfc["day"] = dfc["video_start_timestamp"].dt.tz_convert(tz).dt.floor("D")
    data_days = dfc[["hive", "day"]].drop_duplicates().assign(has_data=True)

    # time-weighted hourly average counts (all three metrics at once)
    df_hour = fc.get_feedercam_hourly_average(
        dfc, dfweather_hourly, video_duration_seconds=video_duration_seconds, hives=list(bd.hives)
    )

    # mask hours on no-data days -> NaN so gaps render as line breaks (not zeros)
    df_hour["day"] = df_hour["hour"].dt.floor("D")
    df_hour = df_hour.merge(data_days, on=["hive", "day"], how="left")
    for metric, _ in metrics:
        df_hour.loc[df_hour["has_data"].isna(), metric] = np.nan

    for metric, mlabel in metrics:
        fig, axes = plt.subplots(5, 1, figsize=(25, 10), sharex=True)
        for ax in axes[1:4]:
            ax.sharey(axes[0])

        for ax, hive, color in zip(axes[:4], bd.hives, sns.color_palette(n_colors=4)):
            dfsel = df_hour[df_hour["hive"] == hive].sort_values("hour")
            y = np.ma.masked_invalid(dfsel[metric].to_numpy())
            ax.plot(dfsel["hour"].to_numpy(), y, color=color, linewidth=1.5)

            bp.shade_treatments(
                ax,
                treat_df[treat_df["feeder"] == f"Feeder {hive}"],
                color="red",
                alpha=0.2,
            )
            ax.set_ylabel(f"{mlabel}\nper hour", fontsize=14)
            ax.text(
                0.01, 0.95, f"Hive {hive}",
                transform=ax.transAxes, ha="left", va="top", fontsize=14,
                bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"),
            )

        weather_ax = axes[-1]
        bp.plot_temp_and_precip(weather_ax, dfweather_hourly)
        bp.plot_sun_thirdaxis(weather_ax, dfweather_hourly)

        bp.common_plot_formatting(axes, start, end, skip=2)
        plt.suptitle(f"{mlabel} counts per hour - {which} cam (CLAHE)", fontsize=20, y=0.99)
        plt.tight_layout()
        plt.show()

## Daily average counts per hive (CLAHE only)

In [ ]:
for which, dfc in [("feeder", dfcounts_feeder), ("exit", dfcounts_exit)]:
    daily = (
        dfc.groupby(["hive", "date"], as_index=False)
        .agg(
            avg_total=("totalcounts", "mean"),
            avg_tagged=("taggedcounts", "mean"),
            avg_untagged=("untaggedcounts", "mean"),
            videos=("video_start_timestamp", "nunique"),
        )
    )

    for suptitle, quantity in zip(
        ["Total counts", "Tagged counts", "Untagged counts"],
        ["avg_total", "avg_tagged", "avg_untagged"],
    ):
        fig, ax = plt.subplots(figsize=(12, 4))
        sns.lineplot(
            data=daily, x="date", y=quantity, hue="hive",
            ax=ax, marker="o", markersize=5,
        )
        ax.set_title(f"{suptitle} - {which} cam (CLAHE)")
        ax.set_ylabel("Avg per video")
        ax.set_xlabel("Date")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
        plt.tight_layout()
        plt.show()